# Delta Lake Optimization & Performance

Practical techniques to maintain Delta tables and keep queries fast: OPTIMIZE, VACUUM, Z-ORDER, Liquid Clustering, Deletion Vectors, Predictive Optimization and query plan analysis.

**Prerequisites:** 03 — Delta Lake (Day 1)

## Learning Objectives

After this module you can:

- Diagnose the small files problem and fix it with `OPTIMIZE` (compaction)
- Manage the table lifecycle with `VACUUM` (retention, `DRY RUN`, safety)
- Apply Z-ORDER and Liquid Clustering for data skipping, and explain why partitioning is rarely the answer today
- Enable Automatic Liquid Clustering (`CLUSTER BY AUTO`) and explain how predictive optimization automates `OPTIMIZE` and `VACUUM`
- Read query plans with `EXPLAIN` and explain deletion vectors
- Diagnose data skew (Query Profile on serverless, Spark UI on classic compute) and fix it with AQE or salting

## Setup

Run the course setup first. It defines the catalog and schema names that the demo tables use.

In [0]:
%run ../../setup/00_setup

### Configuration

In [0]:
# All demo tables go to the bronze schema of the course catalog
from pyspark.sql import functions as F
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {BRONZE_SCHEMA}")
print(f"Current schema: {CATALOG}.{BRONZE_SCHEMA}")

## Optimization Techniques

As a table grows, queries slow down for two reasons. Every small file costs metadata and an open call, so too many small files make each query slower. A data layout that does not match the filters makes the engine read files it could skip.

| Technique | Description | When to Use |
|-----------|-------------|-------------|
| OPTIMIZE | Compacts small files into larger ones | After many small writes |
| Partitioning | Physical data separation by column values | Very large tables with a low-cardinality filter column |
| Z-ORDER | Co-locates related data for better pruning | Frequently filtered columns |
| Liquid Clustering | Modern alternative to partitioning + Z-ORDER | New tables (recommended) |

<img src="../../../assets/images/b784d6b9500944dcad315b7d75c5912b.webp" width="800">

> **Note:** *tabela* and *kolumna* in the image are Polish for table and column. `OPTIMIZE` compacts small files into larger ones (bin-packing). It does not compress the data.

**Optimization: syntax reference**

| Command | Syntax | Description |
|---------|--------|-------------|
| `OPTIMIZE` | `OPTIMIZE table` | Compact small files (target ~1 GB per file) |
| `OPTIMIZE + ZORDER` | `OPTIMIZE table ZORDER BY (col1, col2)` | Compact + co-locate data (no hard column limit; most effective with 1–4 cols) |
| `VACUUM` | `VACUUM table RETAIN 168 HOURS` | Remove unreferenced files (default: 7 days; on serverless `RETAIN` is ignored unless `0 HOURS`) |
| `VACUUM DRY RUN` | `VACUUM table DRY RUN` | List up to 1,000 files VACUUM would delete, delete nothing |
| `VACUUM 0H` | `ALTER TABLE t SET TBLPROPERTIES ('delta.deletedFileRetentionDuration' = 'interval 0 hours'); VACUUM t RETAIN 0 HOURS` | Removes ALL history. **Demo only.** Works on serverless; on classic compute you can instead disable `spark.databricks.delta.retentionDurationCheck.enabled` |
| `DESCRIBE DETAIL` | `DESCRIBE DETAIL table` | Show file count, size, location, clustering |
| `DESCRIBE HISTORY` | `DESCRIBE HISTORY table` | Show all Delta versions with operation info |

### The Small Files Problem

Streaming jobs, frequent small inserts and many concurrent writers leave many small files. Each file adds metadata to read and a file to open, so queries get slower. Run `OPTIMIZE` after such writes: it compacts the small files into a few large ones.

In [0]:
# Auto-optimize off, so the small files stay.
# DROP first: a rerun must start with fresh files and the default retention
SF = f"{CATALOG}.{BRONZE_SCHEMA}.small_files_demo"
spark.sql(f"DROP TABLE IF EXISTS {SF}")
spark.sql(f"""
CREATE TABLE {SF} (
    id INT,
    data STRING,
    created_at TIMESTAMP
)
TBLPROPERTIES (
    delta.autoOptimize.optimizeWrite = false,
    delta.autoOptimize.autoCompact = false
)
""")
print(f"Created {SF}")

In [0]:
# 10,000 rows for up to 5,000 tiny files (empty partitions write none, expect a few thousand)
total_files, rows_per_file = 5000, 2
total_rows = total_files * rows_per_file
df = (spark.range(0, total_rows)
      .withColumn("id", (F.col("id") % 100 + 1).cast("int"))
      .withColumn("data", F.expr("uuid()"))
      .withColumn("created_at", F.current_timestamp()))

In [0]:
# repartition(5000) asks for 5,000 files in one transaction; empty partitions write no file
df.repartition(total_files).write.mode("append").saveAsTable(SF)

In [0]:
# BEFORE OPTIMIZE: thousands of tiny files, no partition columns
display(spark.sql(f"DESCRIBE DETAIL {SF}").select("numFiles", "sizeInBytes", "partitionColumns"))

In [0]:
# OPTIMIZE compacts them: see numFilesRemoved and numFilesAdded
display(spark.sql(f"OPTIMIZE {SF}").select("metrics.numFilesAdded", "metrics.numFilesRemoved"))

In [0]:
# AFTER OPTIMIZE: usually one file here (the current version, VACUUM must not touch it)
display(spark.sql(f"DESCRIBE DETAIL {SF}").select("numFiles", "sizeInBytes"))

### VACUUM: Remove the Files OPTIMIZE Replaced

The result above shows one file here (about 200 KB; on real tables OPTIMIZE writes files of up to about 1 GB). The thousands of small files are still in storage, because older versions of the table reference them. VACUUM deletes all data files that the current version no longer references, once they were removed from the log longer ago than the retention period. It never touches the files of the current version.

| Table property | Default | What it controls |
|---|---|---|
| `delta.deletedFileRetentionDuration` | 7 days | How long removed data files stay before VACUUM may delete them. Time travel to a version that needs files older than this is rejected |
| `delta.logRetentionDuration` | 30 days | How long the table history (transaction log) is kept: `DESCRIBE HISTORY` and time travel stop there |

Time travel to a version needs both its log entries and its data files, so the shorter of the two limits wins.

- `VACUUM t DRY RUN` deletes nothing and lists at most 1,000 of the files it would delete. VACUUM itself deletes all of them. So in the next cells DRY RUN lists 1,000 files, while VACUUM deletes all the small files that OPTIMIZE replaced (a few thousand, see `numDeletedFiles`). The two numbers differ on purpose.
- On serverless, VACUUM ignores the `RETAIN` argument unless it is `0 HOURS`. To change the retention, set `delta.deletedFileRetentionDuration`.
- On Unity Catalog managed tables, predictive optimization runs VACUUM automatically, so you rarely run it yourself (see *Predictive Optimization* below).

The demo lowers the retention to 0 hours so that VACUUM can delete the files right now. **Demo only.** In production keep 7 days or more, because files that VACUUM deleted cannot come back.

<img src="../../../assets/images/nb06-vacuum-retention.png" alt="After OPTIMIZE the small files stay in storage until VACUUM removes them after the retention: deletedFileRetentionDuration 7 days, logRetentionDuration 30 days, the shorter limit wins for time travel" width="800">

In [ ]:
# DEMO ONLY: lower the file retention to 0 hours so VACUUM may delete the replaced files now
spark.sql(f"""
    ALTER TABLE {SF}
    SET TBLPROPERTIES ('delta.deletedFileRetentionDuration' = 'interval 0 hours')
""")
print(f"{SF}: deletedFileRetentionDuration = 0 hours (demo only)")

In [ ]:
# DRY RUN: lists at most the first 1,000 files VACUUM would delete; nothing is deleted yet
dry_run = spark.sql(f"VACUUM {SF} RETAIN 0 HOURS DRY RUN")
print(f"Files listed by DRY RUN (at most 1,000): {dry_run.count():,}")
display(dry_run.limit(5))

In [ ]:
# The real VACUUM: deletes ALL unreferenced files older than the retention,
# not only the ones DRY RUN listed
files_before = spark.sql(f"DESCRIBE DETAIL {SF}").first()["numFiles"]
spark.sql(f"VACUUM {SF} RETAIN 0 HOURS").collect()  # returns only the table path
# The number of deleted files is in the history: VACUUM END, numDeletedFiles
display(spark.sql(f"DESCRIBE HISTORY {SF}")
        .filter("operation = 'VACUUM END'")
        .select("version", F.col("operationMetrics")["numDeletedFiles"].alias("numDeletedFiles")))

In [ ]:
# A second DRY RUN finds nothing left, and the current version keeps its files and all rows
left = spark.sql(f"VACUUM {SF} RETAIN 0 HOURS DRY RUN").count()
files_after = spark.sql(f"DESCRIBE DETAIL {SF}").first()["numFiles"]
rows = spark.table(SF).count()
print(f"Files left for VACUUM: {left} | rows: {rows}")
print(f"numFiles before/after VACUUM: {files_before}/{files_after}")

In [0]:
# The history records every step: WRITE, OPTIMIZE, SET TBLPROPERTIES,
# VACUUM START / VACUUM END (numDeletedFiles)
display(spark.sql(f"DESCRIBE HISTORY {SF}").select("version", "operation", "operationMetrics"))

### Partitioning

`PARTITIONED BY (col)` stores each value of the column in its own folder. A filter on that column then skips whole folders (partition pruning). This pays off only for very large tables with a low-cardinality filter column, with at least about 1 GB per partition. Too many partitions bring back the small files problem. Changing the partition column means rewriting the table.

Databricks advises against partitioning tables under about 1 TB and recommends Liquid Clustering for new tables. A short partitioning demo and a comparison table come after the Liquid Clustering demo.

<img src="../../../assets/images/nb06-partitioning.png" alt="Partitioning: one folder per region value, WHERE region = 'North' reads one folder and skips the others; a high-cardinality column such as order_id creates the small files problem" width="800">

### Z-ORDER (Data Skipping)

Delta keeps min/max statistics for every column in every file. A query with `WHERE col = x` skips the files whose range cannot contain `x`. `OPTIMIZE ... ZORDER BY (cols)` compacts the files and also sorts the rows along a Z-order curve. Each file then covers a narrow range of the Z-ORDER columns, so more files can be skipped.

- Use it for high-cardinality columns that appear in `WHERE` (customer_id, product_id).
- No hard column limit, but the effect drops beyond 3–4 columns.
- A new `ZORDER BY` rewrites the whole table. It is not incremental.

<img src="../../../assets/images/nb06-file-stats.png" alt="Data skipping reads min/max per file: before ZORDER the customer_id ranges overlap and every file is read, after OPTIMIZE ZORDER BY one file is read and three are skipped" width="800">

In [0]:
# Auto-optimize off. DEMO ONLY: a 4 MB target file size, so OPTIMIZE leaves several files
# to skip. The default target (~1 GB) would give one file
ZT = f"{CATALOG}.{BRONZE_SCHEMA}.sales_zorder_demo"
spark.sql(f"DROP TABLE IF EXISTS {ZT}")
spark.sql(f"""
CREATE TABLE {ZT} (
    sale_id STRING,
    customer_id STRING,
    product_id STRING,
    store_id STRING,
    sale_date DATE,
    amount DOUBLE,
    quantity INT
)
TBLPROPERTIES (
    delta.autoOptimize.optimizeWrite = false,
    delta.autoOptimize.autoCompact = false,
    delta.targetFileSize = '4mb'
)
""")
print(f"Created {ZT}")

In [0]:
# 2M random sales (tens of MB): enough for OPTIMIZE ZORDER to write several files
month = (F.rand() * 12 + 1).cast("int")
day = (F.rand() * 28 + 1).cast("int")
sales_df = spark.range(0, 2_000_000).select(
    F.concat(F.lit("SALE-"), F.lpad("id", 8, "0")).alias("sale_id"),
    F.concat(F.lit("CUST"), F.lpad((F.rand() * 1000 + 1).cast("int"), 4, "0")).alias("customer_id"),
    F.concat(F.lit("PROD"), F.lpad((F.rand() * 500 + 1).cast("int"), 3, "0")).alias("product_id"),
    F.concat(F.lit("STORE"), F.lpad((F.rand() * 50 + 1).cast("int"), 2, "0")).alias("store_id"),
    F.make_date(F.lit(2024), month, day).alias("sale_date"),
    (F.rand() * 490 + 10).alias("amount"),
    (F.rand() * 10 + 1).cast("int").alias("quantity"),
)

In [0]:
# 24 files of random rows: every file holds almost every customer and product
sales_df.repartition(24).write.mode("overwrite").saveAsTable(ZT)

In [0]:
# BEFORE Z-ORDER: pick a customer (and a product for the EXPLAIN later),
# then count the files that contain the customer's rows
key = spark.table(ZT).select("customer_id", "product_id").first()
CUST_ID, PROD_ID = key["customer_id"], key["product_id"]
num_files = spark.sql(f"DESCRIBE DETAIL {ZT}").first()["numFiles"]
display(spark.sql(f"""
    SELECT '{CUST_ID}' AS customer_id, {num_files} AS numFiles,
           COUNT(DISTINCT _metadata.file_path) AS files_with_customer
    FROM {ZT}
    WHERE customer_id = '{CUST_ID}'
"""))

In [0]:
# Z-ORDER on the two filter columns
display(spark.sql(f"OPTIMIZE {ZT} ZORDER BY (customer_id, product_id)")
        .select("metrics.numFilesAdded", "metrics.numFilesRemoved"))

In [0]:
# AFTER Z-ORDER: the customer's rows sit in a few files, and min/max statistics skip the others
num_files = spark.sql(f"DESCRIBE DETAIL {ZT}").first()["numFiles"]
display(spark.sql(f"""
    SELECT '{CUST_ID}' AS customer_id, {num_files} AS numFiles,
           COUNT(DISTINCT _metadata.file_path) AS files_with_customer
    FROM {ZT}
    WHERE customer_id = '{CUST_ID}'
"""))
display(spark.sql(f"SELECT * FROM {ZT} WHERE customer_id = '{CUST_ID}'"))

Before Z-ORDER every file contains rows of the customer, so the query reads all of them. After Z-ORDER only a few files do, and Delta skips the rest by their min/max statistics. To see the exact files read and skipped: **See performance** under the code above → in the list, click the `SELECT *` statement (line 9; the line 3 statement looks the same in the list) → **See query profile** → the Scan node shows **Files read** and **Files pruned**.

<img src="../../../assets/images/ss-04-query-profile.png" width="800">

### Liquid Clustering

Liquid Clustering replaces partitioning and Z-ORDER for new tables. You declare the clustering keys on the table, and every `OPTIMIZE` then clusters the new data incrementally. You can change the keys with `ALTER TABLE ... CLUSTER BY` without rewriting the existing data. A table uses either liquid clustering or partitioning and Z-ORDER, not both.

| Operation | Syntax |
|-----------|--------|
| Create with clustering | `CREATE TABLE t (...) CLUSTER BY (col1, col2)` (max 4 keys) |
| Add to / change on an existing table | `ALTER TABLE t CLUSTER BY (col1)`, then `OPTIMIZE t` |
| Let Databricks choose the keys | `CLUSTER BY AUTO` |
| Remove clustering | `ALTER TABLE t CLUSTER BY NONE` |
| Apply the clustering | `OPTIMIZE t` (no `ZORDER BY`) |
| Check | `DESCRIBE DETAIL t`: `clusteringColumns` |

With Automatic Liquid Clustering (`CLUSTER BY AUTO`), predictive optimization picks the keys from the queries that run on the table. It changes the keys when the queries change.

<img src="../../../assets/images/nb06-table-layout.png" alt="Three ways to lay out a table: PARTITIONED BY, ZORDER BY and CLUSTER BY (liquid clustering, recommended for new tables), with the limits of each; one table uses liquid clustering or partitioning and ZORDER, not both" width="800">

In [0]:
# Clustered by (customer_id, region). Auto-optimize off and a 4 MB target (DEMO ONLY),
# so OPTIMIZE leaves several clustered files
LT = f"{CATALOG}.{BRONZE_SCHEMA}.sales_liquid_clustering"
spark.sql(f"DROP TABLE IF EXISTS {LT}")  # a rerun must not keep CLUSTER BY AUTO from the last run
spark.sql(f"""
CREATE TABLE {LT} (
    sale_id STRING,
    customer_id STRING,
    product_id STRING,
    region STRING,
    sale_date DATE,
    amount DOUBLE,
    quantity INT
)
CLUSTER BY (customer_id, region)
TBLPROPERTIES (
    delta.autoOptimize.optimizeWrite = false,
    delta.autoOptimize.autoCompact = false,
    delta.targetFileSize = '4mb'
)
""")
print(f"Created {LT}")

In [0]:
# One batch of random sales. Serverless compacts a single write into ~1 file,
# so the next cell appends several batches
regions = F.array(*[F.lit(r) for r in ["North", "South", "East", "West", "Central"]])

def make_batch(b, rows=200_000):
    customer = F.lpad((F.rand() * 500 + 1).cast("int"), 4, "0")
    product = F.lpad((F.rand() * 200 + 1).cast("int"), 3, "0")
    return spark.range(b * rows, (b + 1) * rows).select(
        F.concat(F.lit("SALE-"), F.lpad("id", 8, "0")).alias("sale_id"),
        F.concat(F.lit("CUST"), customer).alias("customer_id"),
        F.concat(F.lit("PROD"), product).alias("product_id"),
        F.element_at(regions, (F.rand() * 5 + 1).cast("int")).alias("region"),
        F.date_add(F.to_date(F.lit("2024-01-01")), (F.rand() * 364).cast("int")).alias("sale_date"),
        F.round(F.rand() * 490 + 10, 2).alias("amount"),
        (F.rand() * 10 + 1).cast("int").alias("quantity"))

In [0]:
# 10 separate appends: a fragmented table that is not clustered yet
for i in range(10):
    make_batch(i).write.mode("append").saveAsTable(LT)
display(spark.sql(f"DESCRIBE DETAIL {LT}").select("numFiles", "sizeInBytes", "clusteringColumns"))

In [0]:
# OPTIMIZE applies the clustering keys from the table definition, no ZORDER BY needed.
# One row per OPTIMIZE pass: the first is the clustering, later passes find nothing left to do (zeros)
display(spark.sql(f"OPTIMIZE {LT}").select("metrics.numFilesAdded", "metrics.numFilesRemoved"))

In [0]:
# Fewer, clustered files; clusteringColumns = [customer_id, region]
display(spark.sql(f"DESCRIBE DETAIL {LT}").select("numFiles", "sizeInBytes", "clusteringColumns"))

The result above shows fewer files and `clusteringColumns` with both keys.

The `OPTIMIZE` clustered the files by the explicit keys `(customer_id, region)`, so you can measure the effect right away. That is why `CLUSTER BY AUTO` comes *after* `OPTIMIZE` here. With `CLUSTER BY AUTO`, predictive optimization chooses the keys later and **asynchronously**, from observed query patterns. An `OPTIMIZE` right after you enable it would not show a clustering effect yet.

In [0]:
# Switch the same table to Automatic Liquid Clustering. Only metadata changes, no files
# are rewritten now. Look for clusterByAuto = true: predictive optimization picks the keys
# later, from real queries
spark.sql(f"ALTER TABLE {LT} CLUSTER BY AUTO")
props = spark.sql(f"SHOW TBLPROPERTIES {LT}")
display(props.filter("key IN ('clusterByAuto', 'clusteringColumns')"))

### Partitioning in practice

The next table holds the same sales rows, partitioned by `region` (5 values), so each region gets its own folder. A filter on the partition column reads only the matching folder, and the plan shows this as `PartitionFilters`. **Demo only:** this table is about 20 MB in 5 files, far below the size at which partitioning pays off.

In [ ]:
# The same sales, partitioned by region (5 values): one folder per value
PT = f"{CATALOG}.{BRONZE_SCHEMA}.sales_partitioned"
spark.sql(f"""
CREATE OR REPLACE TABLE {PT}
PARTITIONED BY (region)
AS SELECT * FROM {LT}
""")
display(spark.sql(f"DESCRIBE DETAIL {PT}")
        .select("partitionColumns", "clusteringColumns", "numFiles", "sizeInBytes"))
display(spark.sql(f"SHOW PARTITIONS {PT}"))

In [ ]:
# A filter on the partition column: look for PartitionFilters in the scan
display(spark.sql(f"""
    EXPLAIN FORMATTED
    SELECT sum(amount) FROM {PT} WHERE region = 'North'
"""))

In the plan above, look for `PartitionFilters` in the scan: the query reads only the `North` folder.

Liquid Clustering replaces partitioning for new tables for three reasons:
- Changing the partition column means rewriting the whole table. `ALTER TABLE ... CLUSTER BY` changes the keys without a rewrite.
- A high-cardinality column (customer, timestamp) creates thousands of folders and brings back the small files problem. Clustering keys work with any cardinality.
- Partitioning and Liquid Clustering do not mix: a table has one or the other.

**Comparison: Partitioning vs Z-ORDER vs Liquid Clustering**

| Feature | Partitioning | Z-ORDER | Liquid Clustering |
|---------|-------------|---------|-------------------|
| When to choose | Low-cardinality columns | High-cardinality filter columns | General purpose (recommended) |
| Data layout | Directory per partition | Co-located in files | Automatic clustering |
| Changing the layout columns | Full table rewrite | New `ZORDER BY` needs a full re-OPTIMIZE rewrite (not incremental) | `ALTER TABLE ... CLUSTER BY` without an immediate rewrite; later OPTIMIZE runs apply the new keys |
| Maintenance | Manual | Manual OPTIMIZE | Automatic with OPTIMIZE |
| Best for | Date/Region filters | Multi-column filters | Evolving workloads |

## Query Plan Analysis with EXPLAIN

Use `EXPLAIN` when a query is slow and you want to see how Spark runs it. The plan shows full scans, shuffles and the filters that make data skipping possible.

| Command | Description |
|---|---|
| `EXPLAIN query` | Basic physical plan |
| `EXPLAIN FORMATTED query` | Detailed plan with metrics |
| `EXPLAIN EXTENDED query` | Logical + physical + optimized plan |

What to look for in the plan:

| In the plan | Sign | What it means |
|---|---|---|
| `PartitionFilters` | [OK] | Partition pruning is active (partitioned tables) |
| `DataFilters` / `PushedFilters` (classic Spark scan) or `DictionaryFilters` / `RequiredDataFilters` (Photon scan on serverless) | [OK] | The predicate is pushed into the scan |
| `BroadcastHashJoin` / `PhotonBroadcastHashJoin` | [OK] | The smaller side (measured after filters) is broadcast, and the other side is not shuffled. Photon sends the broadcast (build) side through a `SinglePartition` / `EXECUTOR_BROADCAST` exchange: that exchange is the broadcast, not a shuffle |
| `SortMergeJoin` / `PhotonShuffledHashJoin` | [!] | Both sides are shuffled (expensive) |
| Scan without filters | [!] | Full table scan |

> **EXPLAIN does not show data skipping.** It shows which filters are pushed down, not how many files were skipped. Files and bytes read vs pruned are in the **Query Profile** (serverless: "See performance" under the cell) or the Spark UI SQL tab (classic).

In [0]:
# EXPLAIN on the Z-ORDERed table: the pushed filters (DataFilters on classic,
# DictionaryFilters / RequiredDataFilters on Photon). Skipped files are only in the Query Profile
display(spark.sql(f"""
    EXPLAIN FORMATTED
    SELECT * FROM {ZT}
    WHERE customer_id = '{CUST_ID}' AND product_id = '{PROD_ID}'
"""))

In the plan above, find the pushed filters in the scan node. `EXPLAIN` does not read files, so the files read and skipped are only in the Query Profile of the Z-ORDER query in the Z-ORDER section.

## Deletion Vectors

Deletion vectors make DELETE, UPDATE and MERGE faster. Delta Lake marks rows as deleted in a separate bitmap file and does not rewrite whole Parquet files.

- On DELETE or UPDATE, Delta Lake writes a **deletion vector file** instead of rewriting the data files.
- The data files stay unchanged until the next OPTIMIZE or REORG.
- Reads filter out the deleted rows automatically, using the deletion vector.

<img src="../../../assets/images/42198416fc184ab18e07b5c0bcd29468.webp" width="800">

| Aspect | Without Deletion Vectors | With Deletion Vectors |
|--------|------------------------|----------------------|
| DELETE speed | Slow (rewrite files) | Fast (mark in vector) |
| Storage during DELETE | Temporary 2x | Minimal overhead |
| Read performance | Normal | Slight overhead (filter) |
| Cleanup | Not needed | Applied at the next OPTIMIZE, or with `REORG TABLE ... APPLY (PURGE)` |

```sql
-- Enable deletion vectors on a table
ALTER TABLE my_table SET TBLPROPERTIES ('delta.enableDeletionVectors' = true);

-- Purge deletion vectors (rewrite files)
REORG TABLE my_table APPLY (PURGE);
```

Deletion vectors are on by default for new tables in Databricks. DML operations get faster, but reads may have a slight overhead until the next OPTIMIZE or REORG rewrites the files.

## Predictive Optimization

Predictive optimization runs table maintenance for you on Unity Catalog managed tables, so you do not schedule `OPTIMIZE` and `VACUUM` jobs yourself.

![ae9f6138b2354c91a096cfd314482a](../../../assets/images/ss-04-predictive-opt.png "../../../assets/images/ss-04-predictive-opt.png")

- Enabled by default for all accounts, and requires the Premium plan. New accounts have had it since 11 November 2024, and the rollout to existing accounts finished by 1 July 2025.
- Runs `OPTIMIZE` (file compaction), `VACUUM` (removes files the table no longer references) and `ANALYZE` (statistics) when a table needs them.
- Does not run Z-ORDER: an `OPTIMIZE` started by predictive optimization does not apply `ZORDER BY`.
- No SLA. Databricks queues the operations; records typically appear in the history table within 12 hours, so you cannot watch them during the class.
- Billed as serverless jobs compute.
- Managed at catalog, schema, or table level.

```sql
-- Enable at schema level (inherits the account default when not set)
ALTER SCHEMA my_catalog.my_schema
ENABLE PREDICTIVE OPTIMIZATION;

-- Disable for a specific table (e.g., staging tables)
ALTER TABLE my_catalog.my_schema.staging_temp
DISABLE PREDICTIVE OPTIMIZATION;

-- Check optimization history
SELECT start_time, end_time, table_name, operation_type, operation_status
FROM system.storage.predictive_optimization_operations_history
WHERE catalog_name = 'my_catalog'
ORDER BY start_time DESC;
```

| Feature | Manual Maintenance | Predictive Optimization |
|---------|-------------------|------------------------|
| OPTIMIZE | Run manually / schedule | Automatic |
| VACUUM | Run manually / schedule | Automatic |
| ANALYZE | Run manually / schedule | Automatic |
| Tuning | DBA responsibility | Databricks decides from table usage and health |
| Cost | Compute cost on schedule | Billed as serverless jobs compute |
| Default | Must configure | On by default for all accounts |

Predictive optimization works only on Unity Catalog managed tables. For staging or temporary tables that you drop often, consider disabling it to avoid unnecessary runs.

<img src="../../../assets/images/nb06-predictive-optimization.png" alt="Predictive optimization: ENABLE, DISABLE or inherit at catalog, schema and table level, UC managed tables only, queues OPTIMIZE, VACUUM and ANALYZE but not ZORDER BY" width="800">

## Data Skew and Salting

Exam guide objective 6.3 and its sample question 1 cover a skewed `groupBy` with AQE on, fixed by salting.

### What Data Skew Is

Data skew means that data is spread unevenly across partitions. One executor processes 10× or 100× more rows than the others. This single "straggler" task sets the duration of the whole stage.

Why it matters:
- Spark runs all tasks in parallel, but it **waits for the slowest task** before the stage ends.
- One skewed partition leaves all other executors idle.
- A typical symptom: the stage finishes in seconds, except one task that runs for minutes.

Common causes of skew:
- Joining or grouping on a column with a dominant value (for example a top customer, `NULL` or one popular product)
- Low-cardinality `groupBy` keys
- Badly chosen partition columns

#### How to See Skew

- On serverless there is no Spark UI. Open **See performance** under the cell → **Query Profile**, and find the operator with outsized rows or time.
- On classic compute, open Spark UI → **Stages** → the slowest stage → **Summary Metrics**. Skew shows as a **Max** task duration or input size far above the **Median** (3× or more). In the task timeline, one bar is much longer than the rest.

#### Adaptive Query Execution (AQE)

Adaptive Query Execution (AQE) is a Spark framework that re-optimizes query plans at runtime. It uses statistics collected while the query runs.

AQE is enabled by default on Databricks (`spark.sql.adaptive.enabled = true`) and always on for serverless compute.

| Feature | Config flag | What it does |
|---------|------------|-------------- |
| Coalescing small partitions | `spark.sql.adaptive.coalescePartitions.enabled` (default: true) | After shuffle, merges small partitions to reduce task overhead |
| Skew join optimization | `spark.sql.adaptive.skewJoin.enabled` (default: true) | Splits oversized partitions and replicates the matching side to fix skew |
| Dynamic join strategy (sort-merge → broadcast) | `spark.databricks.adaptive.autoBroadcastJoinThreshold` (Databricks, default 30 MB; OSS: `spark.sql.adaptive.autoBroadcastJoinThreshold`) | Converts a sort-merge join to a broadcast hash join at runtime when one side turns out to be small enough |
| Local shuffle reader | `spark.sql.adaptive.localShuffleReader.enabled` (default: true) | After such a conversion, reads the already-written shuffle files locally instead of re-shuffling |
| Empty relation propagation | `spark.databricks.adaptive.emptyRelationPropagation.enabled` (default: true) | Short-circuits joins/aggregations when one side is empty at runtime |

On serverless these keys are managed for you, and setting them raises `CONFIG_NOT_AVAILABLE`. You can set only a short allow-list, for example `spark.sql.shuffle.partitions` (serverless shows `auto`).

To check that AQE is active, run a join or an aggregation. Then look for `AQEShuffleRead` nodes or a runtime-converted `BroadcastHashJoin` in the final plan, in the Query Profile (serverless) or Spark UI → SQL tab (classic).

In [0]:
# Skewed data: customer_id = 99 is one large enterprise with 900k of 1M orders (90%)
df_skewed = spark.range(1_000_000).select(
    F.when(F.col("id") < 900_000, F.lit(99)).otherwise(F.col("id") % 50).alias("customer_id"),
    F.col("id").alias("order_id"),
    (F.col("id") % 500).cast("double").alias("amount"),
)

In [0]:
# Key distribution: customer 99 has 900,000 rows, customers 0-49 have 2,000 each
display(df_skewed.groupBy("customer_id").count().orderBy(F.desc("count")))

In [0]:
# Repartition by the skewed key: one partition (= one task) holds ~900k rows,
# the others a few thousand
df_partitioned = df_skewed.repartition(20, "customer_id")
partition = F.spark_partition_id().alias("partition")
display(df_partitioned.groupBy(partition).count().orderBy("partition"))

In [0]:
# A real job on the skewed partitions. On classic compute the Spark UI stage shows one long task;
# on serverless the Query Profile shows operator times only, so the partition table above is the evidence
display(df_partitioned.groupBy("customer_id").agg(F.sum("amount")))

#### How to Fix Data Skew

In the job above, one task processes about 900k rows and the others a few thousand. The fix depends on the operation:

| Technique | When to use |
|-----------|------------|
| AQE skew join (automatic) | Sort-merge join on a skewed key. On by default, always on in serverless; no code change |
| Salting | Skewed `groupBy`, or a join where AQE does not help |
| Filter the dominant key | Process the dominant value separately, then union it back |
| `repartition(n)` | Even spread before a heavy step when no single value dominates |

Salting appends a random number to the key (`key + '_' + 0..N-1`). You aggregate on the salted key, then strip the salt and aggregate again.

<img src="../../../assets/images/nb06-skew-salting.png" alt="Data skew: customer 99 holds 900k of 1M orders, so one task becomes a straggler; salting splits the hot key into 99_0 to 99_9 and aggregates in two steps" width="800">

AQE skew handling splits skewed partitions in **joins** only. It does not help a skewed `groupBy`, so the next three cells salt the key.

> On serverless, AQE is always on and you cannot change its settings. `spark.conf.set` on keys such as `spark.sql.adaptive.enabled` or `spark.sql.autoBroadcastJoinThreshold` raises `CONFIG_NOT_AVAILABLE`.

In [0]:
# [Fix 1/3] Salt the key: customer 99 becomes 99_0 ... 99_9
SALT_FACTOR = 10
salt = (F.rand() * SALT_FACTOR).cast("int").cast("string")
df_salted = df_skewed.withColumn(
    "customer_id_salted",
    F.concat(F.col("customer_id").cast("string"), F.lit("_"), salt))
salted_keys = df_salted.filter("customer_id = 99").select("customer_id_salted").distinct()
display(salted_keys.orderBy("customer_id_salted"))

In [0]:
# [Fix 2/3] Partition sizes on the salted key: the ~900k straggler is split up.
# Not perfectly even, because 10 big salted keys hashed into 10 partitions can collide
df_fixed = df_salted.repartition(10, "customer_id_salted")
display(df_fixed.groupBy(F.spark_partition_id().alias("partition")).count().orderBy("partition"))

In [0]:
# [Fix 3/3] Aggregate on the salted key, strip the salt, aggregate again: one row per customer
df_aggregated = (df_salted
    .groupBy("customer_id_salted")
    .agg(F.sum("amount").alias("total_amount"), F.count("*").alias("order_count"))
    .withColumn("customer_id", F.split("customer_id_salted", "_")[0].cast("int"))
    .groupBy("customer_id")
    .agg(F.sum("total_amount").cast("decimal(18,2)").alias("total_amount"), F.sum("order_count").alias("order_count")))
display(df_aggregated.orderBy(F.desc("order_count")))

Compare the two partition tables: after salting the largest partition holds far fewer rows than the ~900k of the skewed one. On classic compute the Spark UI stage shows the same as task times; on serverless the Query Profile shows operator times only, so the partition tables are the evidence.

## Summary

| Topic | Key Takeaway |
|---|---|
| OPTIMIZE | Compacts small files. Run after streaming/frequent inserts |
| VACUUM | Deletes unreferenced files older than the retention; `DRY RUN` first |
| Partitioning | Rarely needed: very large tables, low-cardinality column, ≥ 1 GB per partition |
| Z-ORDER | Co-locates data for data skipping on filter columns |
| Liquid Clustering | Modern replacement: incremental, OPTIMIZE still needed but faster |
| Automatic Liquid Clustering | `CLUSTER BY AUTO`: predictive optimization picks and evolves the keys |
| Data Skew | AQE fixes skewed joins automatically; a skewed `groupBy` needs salting |
| Deletion Vectors | Fast DELETE/UPDATE via marking instead of rewriting files |
| Predictive Optimization | Automatic OPTIMIZE, VACUUM and ANALYZE for UC managed tables |

### Quick Reference

| Operation | Command |
|---|---|
| Optimize | `OPTIMIZE table` |
| Z-ORDER | `OPTIMIZE table ZORDER BY (col)` |
| Vacuum | `VACUUM table [DRY RUN]` |
| Deletion Vectors | `ALTER TABLE SET TBLPROPERTIES ('delta.enableDeletionVectors' = true)` |
| Purge DVs | `REORG TABLE ... APPLY (PURGE)` |

> **Note:** Change Data Feed (CDF) is covered in **03 — Delta Lake** (fundamentals) and **BONUS_04 — Streaming Advanced** (incremental ETL usage).

## Cleanup

There is nothing to run here. The demo tables stay in your catalog, and the catalog is removed after the training. To drop them now, run `DROP TABLE IF EXISTS <CATALOG>.bronze.<table>` for `small_files_demo`, `sales_zorder_demo`, `sales_liquid_clustering` and `sales_partitioned`.

## Self-study

Shuffle, broadcast joins and disk spill are not run in class. `lab_troubleshooting` practises them hands-on, and `05a_transformations_pyspark` covers `spark.sql.shuffle.partitions`.

### Shuffle

A shuffle moves data between executors. Every operation that needs the data redistributed triggers one:

| Operation | Shuffle triggered? |
|-----------|-----------------|
| `groupBy` + aggregation | Yes |
| `join` (sort-merge) | Yes |
| `distinct` | Yes |
| `repartition(n)` | Yes |
| `coalesce(n)` | No (narrow) |
| `filter`, `select`, `withColumn` | No (narrow) |
| broadcast join | No |

How to reduce shuffle:

1. Use a broadcast join. If one table is small, force a broadcast to avoid the shuffle entirely.
2. Use partition pruning: filter early to reduce the rows before a join.
3. Choose between coalesce and repartition:
   - `coalesce(n)` reduces partitions without a full shuffle. Use it to shrink before a write.
   - `repartition(n)` does a full shuffle. Use it to increase partitions or to rebalance before heavy operations.

<img src="../../../assets/images/nb06-join-strategies.png" alt="Broadcast hash join copies the small table to every executor with no shuffle of the large side; sort-merge join shuffles and sorts both sides; AQE can switch to broadcast at runtime" width="800">

In [0]:
# Broadcast join: the small products table goes to every executor, the large side is not
# shuffled. Photon (serverless) shows PhotonBroadcastHashJoin. The SinglePartition exchange
# on the small side is the broadcast, not a shuffle
from pyspark.sql.functions import broadcast

orders = (spark.range(1_000_000)
          .withColumnRenamed("id", "order_id")
          .withColumn("product_id", (F.col("order_id") % 100).cast("int")))
products = (spark.range(100)
            .withColumnRenamed("id", "product_id")
            .withColumn("product_name", F.concat(F.lit("Product_"), F.col("product_id"))))
orders.join(broadcast(products), on="product_id").explain(mode="formatted")

### Disk Spilling

Spilling happens when Spark cannot fit shuffle data or sort buffers in executor memory and writes the overflow to disk.

Causes:
- Too few partitions, so each partition is too large to fit in memory
- Executor memory too small for the workload
- Large sort operations (for example `orderBy` on a huge dataset)

How to detect spilling:
- On classic compute, open Spark UI → **Stages** tab → click a stage → scroll to the **Summary Metrics** table.
- Look for the **Spill (Memory)** and **Spill (Disk)** columns. Any non-zero value means spilling.
- On serverless, open the Query Profile. The operator details show **spill** metrics per operator (there is no Spark UI).

How to fix spilling:

| Fix | How |
|-----|-----|
| Increase `spark.sql.shuffle.partitions` | Smaller partitions fit in memory |
| Enable AQE coalesce (on by default) | AQE merges small partitions after shuffle |
| Increase executor memory | Resize cluster node type |
| Reduce broadcast threshold | Avoid accidental large broadcasts |

← <a href="$./05b_transformations_sql">05b — Transformations in SQL</a> | **<a href="$../../..">Course folder (README)</a>** | <a href="$./07_lakeflow_pipelines">07 — Lakeflow Pipelines →</a>

## Self-check (trainer): participants can skip this cell

In [ ]:
# Self-check (trainer): the VACUUM demo on small_files_demo

# The DRY RUN found replaced files: VACUUM logged a VACUUM END with deleted files
deleted = (spark.sql(f"DESCRIBE HISTORY {SF}")
           .filter("operation = 'VACUUM END' AND CAST(operationMetrics['numDeletedFiles'] AS INT) > 0")
           .count())
assert deleted > 0, "no VACUUM END with numDeletedFiles > 0: OPTIMIZE left no replaced files for VACUUM"

# After VACUUM: nothing left to delete, the current version keeps its files and all rows
assert left == 0, f"{left} unreferenced files are still there after VACUUM"
assert files_after == files_before, f"VACUUM changed the files of the current version: {files_before} -> {files_after}"
assert rows == total_rows, f"expected {total_rows} rows, got {rows}"

print("Self-check passed")